In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, ToggleButtons, Play, jslink, HBox, VBox, HTML, Layout, interactive
from IPython.display import display


# ============================================================
# LIMIT CYCLES AS MOTION ON A FINITE STATE GRID
# ============================================================
#
# This notebook demonstrates the finite-state interpretation of
# limit cycles in a quantized second-order recursive digital filter.
#
# The zero-input linear system is
#
#       y[n] = a1 y[n-1] + a2 y[n-2]
#
# while its finite-precision implementation is modeled as
#
#       yq[n] =
#           Q(a1q yq[n-1]) +
#           Q(a2q yq[n-2]).
#
# The state vector is
#
#       q[n] = [y[n-1], y[n]]^T.
#
# The unquantized reference trajectory uses the same stored
# coefficients a1q and a2q, but no arithmetic quantization is
# applied during the recursion.
#
# For a stable linear system, the unquantized trajectory converges
# continuously toward the origin.
#
# The quantized trajectory, however, is restricted to a finite set
# of discrete states. If a previously visited state is reached again,
# deterministic recursion forces the subsequent sequence of states
# to repeat, creating a limit cycle.
#
#
# ============================================================
# IMPORTANT VISUAL DISTINCTION
# ============================================================
#
# BLUE DASHED CURVES:
#
#       Unquantized reference
#
# These trajectories converge toward zero when the linear system is
# stable. They are NOT limit cycles.
#
# ORANGE CURVES / STEMS:
#
#       Quantized response
#
# These may become trapped in a periodic finite-state trajectory.
#
# GREEN SQUARES:
#
#       Detected limit cycle
#
# This distinction is deliberately kept visually explicit throughout
# the notebook.
#
#
# ============================================================
# TIME-EVOLUTION CONTROL
# ============================================================
#
# The Time step n slider determines how much of the trajectory is
# visible.
#
# Both the Time step slider and the Play control start at n = 1.
#
# The Play control advances the trajectory automatically every
# 800 milliseconds.
#
# During automatic playback, all controls that modify the system are
# disabled. Only the Play/Pause control remains active.
#
#
# ============================================================
# DISPLAY ORGANIZATION
# ============================================================
#
# LEFT COLUMN
#
#   Top:
#       Zero-input output sequence.
#
#   Middle:
#       Distance of the state vector from the origin.
#
#   Bottom:
#       Numerical state monitor and cycle information.
#
# RIGHT COLUMN
#
#   Large state-space trajectory.
#
#
# ============================================================
# AUTOMATIC STATE-SPACE ZOOM
# ============================================================
#
# The state-space limits are calculated from the COMPLETE simulated
# trajectory. Therefore, the axes remain fixed during animation.
#
# This prevents distracting rescaling while the trajectory is being
# constructed.
#
#
# ============================================================
# OVERFLOW
# ============================================================
#
# This notebook studies quantization limit cycles only.
#
# The representable interval is
#
#       -1 <= y <= 1 - Delta.
#
# If the recursion attempts to leave this interval, the simulation
# stops and reports overflow.
#
# Overflow-induced limit cycles are studied separately.
#
# ============================================================


# ------------------------------------------------------------
# Quantization functions
# ------------------------------------------------------------

def round_quantizer(x, Delta):

    x_array = np.asarray(x, dtype=float)

    q_index = np.where(x_array >= 0.0, np.floor(x_array / Delta + 0.5), np.ceil(x_array / Delta - 0.5))

    result = Delta * q_index

    if np.ndim(result) == 0:
        return float(result)

    return result


def magnitude_truncation_quantizer(x, Delta):

    x_array = np.asarray(x, dtype=float)

    result = Delta * np.trunc(x_array / Delta)

    if np.ndim(result) == 0:
        return float(result)

    return result


def quantize_value(x, Delta, mode):

    if mode == 'Rounding':
        return round_quantizer(x, Delta)

    return magnitude_truncation_quantizer(x, Delta)


def stored_value(x, Delta, mode):

    return quantize_value(x, Delta, mode)


# ------------------------------------------------------------
# Pole calculation and stability
# ------------------------------------------------------------

def calculate_poles(a1, a2):

    return np.roots([1.0, -a1, -a2])


def is_linearly_stable(a1, a2):

    poles = calculate_poles(a1, a2)

    return np.all(np.abs(poles) < 1.0 - 1e-12)


# ------------------------------------------------------------
# Unquantized reference
# ------------------------------------------------------------

def simulate_linear_reference(a1q, a2q, ym1, y0, samples):

    y = np.zeros(samples + 2)

    y[0] = ym1

    y[1] = y0

    for n in range(2, samples + 2):

        y[n] = a1q * y[n - 1] + a2q * y[n - 2]

    return y


# ------------------------------------------------------------
# Quantized second-order recursion
# ------------------------------------------------------------

def simulate_quantized_system(a1, a2, K, ym1, y0, samples, mode):

    Delta = 2.0**(-K)

    minimum_value = -1.0

    maximum_value = 1.0 - Delta

    a1q = stored_value(a1, Delta, mode)

    a2q = stored_value(a2, Delta, mode)

    ym1q = stored_value(ym1, Delta, mode)

    y0q = stored_value(y0, Delta, mode)

    y = np.zeros(samples + 2)

    y[0] = ym1q

    y[1] = y0q

    overflow_index = None

    for n in range(2, samples + 2):

        product_1 = quantize_value(a1q * y[n - 1], Delta, mode)

        product_2 = quantize_value(a2q * y[n - 2], Delta, mode)

        candidate = product_1 + product_2

        if candidate < minimum_value - 1e-12 or candidate > maximum_value + 1e-12:

            overflow_index = n - 1

            y = y[:n]

            break

        y[n] = stored_value(candidate, Delta, mode)

    return y, a1q, a2q, ym1q, y0q, overflow_index


# ------------------------------------------------------------
# State representation
# ------------------------------------------------------------

def state_key(a, b, Delta):

    return (int(np.round(a / Delta)), int(np.round(b / Delta)))


# ------------------------------------------------------------
# Detect the first repeated state
# ------------------------------------------------------------

def detect_repeated_state(y, Delta):

    visited = {}

    if len(y) < 2:
        return None, None, None, []

    states = []

    for n in range(1, len(y)):

        key = state_key(y[n - 1], y[n], Delta)

        states.append(key)

        if key in visited:

            first_index = visited[key]

            second_index = n - 1

            period = second_index - first_index

            cycle_states = states[first_index:second_index]

            return first_index, second_index, period, cycle_states

        visited[key] = n - 1

    return None, None, None, []


# ------------------------------------------------------------
# Convert integer states to physical amplitudes
# ------------------------------------------------------------

def physical_cycle_states(cycle_states, Delta):

    return [(state[0] * Delta, state[1] * Delta) for state in cycle_states]


# ------------------------------------------------------------
# Automatic state-space limits
# ------------------------------------------------------------

def automatic_state_limits(x_linear, y_linear, x_quantized, y_quantized, Delta):

    all_x = np.concatenate((np.asarray(x_linear), np.asarray(x_quantized), np.array([0.0])))

    all_y = np.concatenate((np.asarray(y_linear), np.asarray(y_quantized), np.array([0.0])))

    x_min = np.min(all_x)

    x_max = np.max(all_x)

    y_min = np.min(all_y)

    y_max = np.max(all_y)

    x_span = x_max - x_min

    y_span = y_max - y_min

    span = max(x_span, y_span, 6.0 * Delta)

    margin = max(2.0 * Delta, 0.15 * span)

    center_x = 0.5 * (x_min + x_max)

    center_y = 0.5 * (y_min + y_max)

    half_span = 0.5 * span + margin

    x_lower = max(-1.0, center_x - half_span)

    x_upper = min(1.0, center_x + half_span)

    y_lower = max(-1.0, center_y - half_span)

    y_upper = min(1.0, center_y + half_span)

    final_span = max(x_upper - x_lower, y_upper - y_lower)

    center_x = 0.5 * (x_lower + x_upper)

    center_y = 0.5 * (y_lower + y_upper)

    x_lower = max(-1.0, center_x - final_span / 2.0)

    x_upper = min(1.0, center_x + final_span / 2.0)

    y_lower = max(-1.0, center_y - final_span / 2.0)

    y_upper = min(1.0, center_y + final_span / 2.0)

    return x_lower, x_upper, y_lower, y_upper


# ------------------------------------------------------------
# CSS
# ------------------------------------------------------------

style_html = HTML("""
<style>

.sg-root {
    width: 960px;
    max-width: 960px;
    font-family: Arial, sans-serif;
}

.sg-header {
    background: #303845;
    color: white;
    padding: 8px 14px;
    border-radius: 7px 7px 0 0;
    font-size: 19px;
    font-weight: bold;
}

.sg-intro {
    background: #f5f7f9;
    border: 1px solid #d4d9df;
    border-top: none;
    padding: 6px 12px;
    border-radius: 0 0 7px 7px;
    font-size: 12px;
    line-height: 1.35;
    margin-bottom: 6px;
}

.sg-accent {
    font-weight: bold;
    color: #384f67;
}

.sg-controls-title {
    font-size: 12.5px;
    font-weight: bold;
    margin: 0 0 3px 3px;
    color: #303845;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Header
# ------------------------------------------------------------

header_html = HTML("""
<div class="sg-root">

    <div class="sg-header">
        Limit Cycles as Motion on a Finite State Grid
    </div>

    <div class="sg-intro">
        <span class="sg-accent">How it works:</span>
        the unquantized state converges toward the origin, while the quantized recursion can visit only discrete states.
        <span class="sg-accent">What to observe:</span>
        advance time and watch the quantized trajectory eventually close into a periodic limit cycle.
    </div>

</div>
""")


# ------------------------------------------------------------
# Main plotting function
# ------------------------------------------------------------

def plot_state_grid(a1=0.75, a2=-0.50, K=4, ym1=0.50, y0=0.375, samples=50, time_step=1, mode='Rounding'):

    Delta = 2.0**(-K)


    # --------------------------------------------------------
    # Complete simulations
    # --------------------------------------------------------

    yq_full, a1q, a2q, ym1q, y0q, overflow_index = simulate_quantized_system(a1, a2, K, ym1, y0, samples, mode)

    y_linear_full = simulate_linear_reference(a1q, a2q, ym1q, y0q, samples)


    # --------------------------------------------------------
    # Current time step
    # --------------------------------------------------------

    maximum_step = min(samples, len(yq_full) - 1)

    current_step = int(np.clip(time_step, 1, maximum_step))


    # --------------------------------------------------------
    # System properties
    # --------------------------------------------------------

    poles = calculate_poles(a1q, a2q)

    linear_stable = is_linearly_stable(a1q, a2q)

    full_first_repeat, full_second_repeat, full_period, full_cycle_states_integer = detect_repeated_state(yq_full, Delta)


    # --------------------------------------------------------
    # Visible trajectories
    # --------------------------------------------------------

    yq = yq_full[:current_step + 1]

    y_linear = y_linear_full[:current_step + 1]

    state_x_q = yq[:-1]

    state_y_q = yq[1:]

    state_x_linear = y_linear[:-1]

    state_y_linear = y_linear[1:]


    # --------------------------------------------------------
    # Visible limit-cycle detection
    # --------------------------------------------------------

    first_repeat, second_repeat, period, cycle_states_integer = detect_repeated_state(yq, Delta)

    cycle_states = physical_cycle_states(cycle_states_integer, Delta)


    # --------------------------------------------------------
    # Current status
    # --------------------------------------------------------

    if overflow_index is not None and current_step >= overflow_index:

        status = 'OVERFLOW REACHED'

        status_detail = f'Overflow reached at n = {overflow_index}.'

    elif period is None:

        status = 'TRANSIENT EVOLUTION'

        status_detail = 'No repeated quantized state has been reached yet.'

    elif period == 1 and len(cycle_states) == 1 and np.isclose(cycle_states[0][0], 0.0) and np.isclose(cycle_states[0][1], 0.0):

        status = 'ZERO STATE REACHED'

        status_detail = 'The quantized recursion has reached zero.'

    elif period == 1:

        status = 'PERIOD-1 LIMIT CYCLE'

        status_detail = 'A non-zero quantized state is repeating.'

    else:

        status = f'PERIOD-{period} LIMIT CYCLE'

        status_detail = f'The first repeated quantized state appears after {first_repeat} transitions.'


    # ========================================================
    # MAIN FIGURE
    # ========================================================

    fig = plt.figure(figsize=(13.2, 7.6))

    grid = fig.add_gridspec(3, 2, width_ratios=[1.05, 1.25], height_ratios=[1.0, 1.0, 1.28], wspace=0.27, hspace=1.05)

    ax_output = fig.add_subplot(grid[0, 0])

    ax_distance = fig.add_subplot(grid[1, 0])

    ax_monitor = fig.add_subplot(grid[2, 0])

    ax_state = fig.add_subplot(grid[:, 1])


    # ========================================================
    # PANEL 1 — ZERO-INPUT OUTPUT SEQUENCE
    # ========================================================

    output_q = yq[1:]

    output_linear = y_linear[1:]

    n_q = np.arange(1, len(yq))


    # --------------------------------------------------------
    # Unquantized reference
    # --------------------------------------------------------

    ax_output.plot(
        n_q,
        output_linear,
        '--',
        color='tab:blue',
        linewidth=1.6,
        label='Unquantized output — decays to zero'
    )


    # --------------------------------------------------------
    # Quantized output
    # --------------------------------------------------------

    markerline, stemlines, baseline = ax_output.stem(
        n_q,
        output_q,
        linefmt='tab:orange',
        markerfmt='o',
        basefmt=' '
    )

    plt.setp(
        stemlines,
        color='tab:orange',
        linewidth=1.2
    )

    plt.setp(
        markerline,
        color='tab:orange',
        markerfacecolor='tab:orange',
        markeredgecolor='tab:orange',
        markersize=4.3
    )


    # --------------------------------------------------------
    # Dummy legend entry for the quantized stems
    # --------------------------------------------------------

    ax_output.plot(
        [],
        [],
        'o-',
        color='tab:orange',
        linewidth=1.2,
        markersize=4.3,
        label='Quantized output'
    )


    # --------------------------------------------------------
    # Mark when the eventual quantized cycle begins
    # --------------------------------------------------------

    if full_first_repeat is not None and current_step >= full_first_repeat:

        ax_output.axvline(
            full_first_repeat,
            color='tab:green',
            linestyle=':',
            linewidth=1.5,
            label='Cycle begins'
        )


    ax_output.axhline(
        0.0,
        color='0.45',
        linewidth=0.7
    )


    # --------------------------------------------------------
    # Fixed axes during animation
    # --------------------------------------------------------

    ax_output.set_xlim(
        0,
        samples
    )


    full_output_linear = y_linear_full[1:]

    full_output_quantized = yq_full[1:]


    output_min = min(
        np.min(full_output_linear),
        np.min(full_output_quantized),
        -Delta
    )


    output_max = max(
        np.max(full_output_linear),
        np.max(full_output_quantized),
        Delta
    )


    output_margin = 0.08 * max(
        output_max - output_min,
        Delta
    )


    ax_output.set_ylim(
        output_min - output_margin,
        output_max + output_margin
    )


    ax_output.set_xlabel(
        'Sample index n',
        labelpad=8
    )


    ax_output.set_ylabel(
        'Output y[n]',
        labelpad=6
    )


    ax_output.set_title(
        'Zero-Input Output Sequence',
        fontsize=11
    )


    ax_output.tick_params(
        labelsize=9
    )


    ax_output.grid(
        True,
        linestyle=':',
        alpha=0.28
    )


    ax_output.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.31),
        ncol=2,
        frameon=False,
        fontsize=8.5,
        columnspacing=1.6
    )


    # ========================================================
    # PANEL 2 — DISTANCE FROM THE ORIGIN
    # ========================================================

    radius_q = np.sqrt(
        state_x_q**2 + state_y_q**2
    )


    radius_linear = np.sqrt(
        state_x_linear**2 + state_y_linear**2
    )


    n_radius = np.arange(
        1,
        len(radius_q) + 1
    )


    ax_distance.plot(
        n_radius,
        radius_linear,
        '--',
        color='tab:blue',
        linewidth=1.6,
        label='Unquantized distance'
    )


    ax_distance.plot(
        n_radius,
        radius_q,
        'o-',
        color='tab:orange',
        linewidth=1.4,
        markersize=3.7,
        label='Quantized distance'
    )


    if full_first_repeat is not None and current_step >= full_first_repeat:

        ax_distance.axvline(
            full_first_repeat,
            color='tab:green',
            linestyle=':',
            linewidth=1.5
        )


    ax_distance.set_xlim(
        0,
        samples
    )


    full_state_x_q = yq_full[:-1]

    full_state_y_q = yq_full[1:]


    full_state_x_linear = y_linear_full[:-1]

    full_state_y_linear = y_linear_full[1:]


    full_radius_q = np.sqrt(
        full_state_x_q**2 + full_state_y_q**2
    )


    full_radius_linear = np.sqrt(
        full_state_x_linear**2 + full_state_y_linear**2
    )


    radius_max = max(
        np.max(full_radius_q),
        np.max(full_radius_linear),
        Delta
    )


    ax_distance.set_ylim(
        0.0,
        1.08 * radius_max
    )


    ax_distance.set_xlabel(
        'State transition index',
        labelpad=8
    )


    ax_distance.set_ylabel(
        r'$\|\mathbf{q}[n]\|_2$',
        labelpad=6
    )


    ax_distance.set_title(
        'Distance from the Origin',
        fontsize=11
    )


    ax_distance.tick_params(
        labelsize=9
    )


    ax_distance.grid(
        True,
        linestyle=':',
        alpha=0.28
    )


    ax_distance.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.32),
        ncol=2,
        frameon=False,
        fontsize=9,
        columnspacing=1.8
    )


    # ========================================================
    # PANEL 3 — STATE MONITOR
    # ========================================================

    ax_monitor.axis(
        'off'
    )


    pole_1 = poles[0]

    pole_2 = poles[1]


    stability_text = 'STABLE' if linear_stable else 'NOT STABLE'


    monitor_1 = (
        f'STATE MONITOR\n'
        f'────────────────\n'
        f'Quantizer : {mode}\n'
        f'K         : {K}\n'
        f'Delta     : {Delta:.6f}\n'
        f'Time n    : {current_step}'
    )


    monitor_2 = (
        f'STORED COEFFICIENTS\n'
        f'───────────────────\n'
        f'a1 : {a1q:.6f}\n'
        f'a2 : {a2q:.6f}\n'
        f'p1 : {pole_1.real:+.4f}{pole_1.imag:+.4f}j\n'
        f'p2 : {pole_2.real:+.4f}{pole_2.imag:+.4f}j\n'
        f'System : {stability_text}'
    )


    monitor_3 = (
        f'CURRENT RESULT\n'
        f'───────────────────\n'
        f'Visited : {len(set(zip(state_x_q, state_y_q)))}\n'
        f'Status  : {status}\n'
    )


    if period is not None:

        monitor_3 += (
            f'Start   : {first_repeat}\n'
            f'Period  : {period}'
        )

    else:

        monitor_3 += f'{status_detail}'


    # --------------------------------------------------------
    # Cycle-state list
    # --------------------------------------------------------

    if period is None:

        cycle_text_1 = 'No complete quantized cycle detected at this time step.'

        cycle_text_2 = ''

    else:

        formatted_states = [
            f'{index}: ({state[0]:+.4f}, {state[1]:+.4f})'
            for index, state in enumerate(cycle_states)
        ]


        split_index = int(
            np.ceil(len(formatted_states) / 2.0)
        )


        cycle_text_1 = '    '.join(
            formatted_states[:split_index]
        )


        cycle_text_2 = '    '.join(
            formatted_states[split_index:]
        )


    ax_monitor.text(
        0.00,
        0.98,
        monitor_1,
        transform=ax_monitor.transAxes,
        ha='left',
        va='top',
        fontsize=8.8,
        family='monospace',
        linespacing=1.30
    )


    ax_monitor.text(
        0.34,
        0.98,
        monitor_2,
        transform=ax_monitor.transAxes,
        ha='left',
        va='top',
        fontsize=8.5,
        family='monospace',
        linespacing=1.27
    )


    ax_monitor.text(
        0.72,
        0.98,
        monitor_3,
        transform=ax_monitor.transAxes,
        ha='left',
        va='top',
        fontsize=8.5,
        family='monospace',
        linespacing=1.27
    )


    ax_monitor.text(
        0.00,
        0.33,
        'QUANTIZED CYCLE STATES',
        transform=ax_monitor.transAxes,
        ha='left',
        va='top',
        fontsize=8.8,
        family='monospace',
        fontweight='bold'
    )


    ax_monitor.text(
        0.00,
        0.22,
        cycle_text_1,
        transform=ax_monitor.transAxes,
        ha='left',
        va='top',
        fontsize=8.2,
        family='monospace'
    )


    ax_monitor.text(
        0.00,
        0.09,
        cycle_text_2,
        transform=ax_monitor.transAxes,
        ha='left',
        va='top',
        fontsize=8.2,
        family='monospace'
    )


    # ========================================================
    # PANEL 4 — LARGE STATE-SPACE TRAJECTORY
    # ========================================================

    x_lower, x_upper, y_lower, y_upper = automatic_state_limits(
        full_state_x_linear,
        full_state_y_linear,
        full_state_x_q,
        full_state_y_q,
        Delta
    )


    # --------------------------------------------------------
    # Quantization grid
    # --------------------------------------------------------

    grid_start_x = np.ceil(
        x_lower / Delta
    ) * Delta


    grid_end_x = np.floor(
        x_upper / Delta
    ) * Delta


    grid_start_y = np.ceil(
        y_lower / Delta
    ) * Delta


    grid_end_y = np.floor(
        y_upper / Delta
    ) * Delta


    grid_x = np.arange(
        grid_start_x,
        grid_end_x + 0.5 * Delta,
        Delta
    )


    grid_y = np.arange(
        grid_start_y,
        grid_end_y + 0.5 * Delta,
        Delta
    )


    if len(grid_x) <= 40:

        for value in grid_x:

            ax_state.axvline(
                value,
                color='0.70',
                linewidth=0.45,
                alpha=0.25
            )


    if len(grid_y) <= 40:

        for value in grid_y:

            ax_state.axhline(
                value,
                color='0.70',
                linewidth=0.45,
                alpha=0.25
            )


    # --------------------------------------------------------
    # Unquantized trajectory
    # --------------------------------------------------------

    ax_state.plot(
        state_x_linear,
        state_y_linear,
        '--',
        color='tab:blue',
        linewidth=1.8,
        label='Unquantized trajectory — converges to origin'
    )


    # --------------------------------------------------------
    # Quantized trajectory
    # --------------------------------------------------------

    ax_state.plot(
        state_x_q,
        state_y_q,
        'o-',
        color='tab:orange',
        linewidth=1.8,
        markersize=5.0,
        label='Quantized trajectory'
    )


    # --------------------------------------------------------
    # Current state
    # --------------------------------------------------------

    if len(state_x_q) > 0:

        ax_state.plot(
            state_x_q[-1],
            state_y_q[-1],
            'o',
            color='tab:green',
            markersize=10,
            markerfacecolor='none',
            markeredgewidth=2.0,
            label='Current quantized state'
        )


    # --------------------------------------------------------
    # Detected quantized limit cycle
    # --------------------------------------------------------

    if period is not None and len(cycle_states) > 0:

        cycle_x = [
            state[0]
            for state in cycle_states
        ]


        cycle_y = [
            state[1]
            for state in cycle_states
        ]


        if len(cycle_states) > 1:

            cycle_x = cycle_x + [
                cycle_x[0]
            ]


            cycle_y = cycle_y + [
                cycle_y[0]
            ]


        ax_state.plot(
            cycle_x,
            cycle_y,
            's-',
            color='tab:green',
            linewidth=2.8,
            markersize=8,
            label=f'Detected period-{period} limit cycle'
        )


    # --------------------------------------------------------
    # Initial state
    # --------------------------------------------------------

    ax_state.plot(
        full_state_x_q[0],
        full_state_y_q[0],
        'D',
        color='tab:purple',
        markersize=7,
        label='Initial state'
    )


    # --------------------------------------------------------
    # Origin
    # --------------------------------------------------------

    ax_state.plot(
        0.0,
        0.0,
        '+',
        color='0.35',
        markersize=11,
        markeredgewidth=1.8
    )


    ax_state.set_xlim(
        x_lower,
        x_upper
    )


    ax_state.set_ylim(
        y_lower,
        y_upper
    )


    ax_state.set_aspect(
        'equal',
        adjustable='box'
    )


    ax_state.set_xlabel(
        r'$y[n-1]$',
        fontsize=12,
        labelpad=8
    )


    ax_state.set_ylabel(
        r'$y[n]$',
        fontsize=12,
        labelpad=8
    )


    ax_state.set_title(
        f'State-Space Trajectory — Time Step n = {current_step}',
        fontsize=13
    )


    ax_state.tick_params(
        labelsize=10
    )


    ax_state.grid(
        True,
        linestyle=':',
        alpha=0.30
    )


    ax_state.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.08),
        ncol=2,
        frameon=False,
        fontsize=8.8,
        columnspacing=1.3
    )


    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    fig.suptitle(
        'Finite-State Evolution of a Quantized Second-Order IIR Filter',
        fontsize=13
    )


    plt.subplots_adjust(
        left=0.06,
        right=0.985,
        top=0.92,
        bottom=0.06
    )


    plt.show()

    plt.close(fig)


# ============================================================
# CONTROLS
# ============================================================

control_style = {
    'description_width': '58px'
}


a1_slider = FloatSlider(
    value=0.75,
    min=-1.50,
    max=1.50,
    step=0.05,
    description='a1:',
    continuous_update=True,
    readout_format='.2f',
    style=control_style,
    layout=Layout(width='330px')
)


a2_slider = FloatSlider(
    value=-0.50,
    min=-0.95,
    max=0.95,
    step=0.05,
    description='a2:',
    continuous_update=True,
    readout_format='.2f',
    style=control_style,
    layout=Layout(width='330px')
)


K_slider = IntSlider(
    value=4,
    min=2,
    max=8,
    step=1,
    description='Bits K:',
    continuous_update=True,
    style={
        'description_width': '60px'
    },
    layout=Layout(width='270px')
)


ym1_slider = FloatSlider(
    value=0.50,
    min=-0.90,
    max=0.90,
    step=0.05,
    description='y[-1]:',
    continuous_update=True,
    readout_format='.2f',
    style=control_style,
    layout=Layout(width='300px')
)


y0_slider = FloatSlider(
    value=0.375,
    min=-0.90,
    max=0.90,
    step=0.05,
    description='y[0]:',
    continuous_update=True,
    readout_format='.3f',
    style=control_style,
    layout=Layout(width='300px')
)


samples_slider = IntSlider(
    value=50,
    min=20,
    max=120,
    step=5,
    description='Samples:',
    continuous_update=True,
    style={
        'description_width': '65px'
    },
    layout=Layout(width='280px')
)


mode_buttons = ToggleButtons(
    options=[
        'Rounding',
        'Magnitude truncation'
    ],
    value='Rounding',
    description='Quantizer:',
    style={
        'description_width': '75px'
    },
    layout=Layout(width='405px')
)


# ------------------------------------------------------------
# Time control starts at n = 1
# ------------------------------------------------------------

time_slider = IntSlider(
    value=1,
    min=1,
    max=50,
    step=1,
    description='Time step n:',
    continuous_update=True,
    style={
        'description_width': '85px'
    },
    layout=Layout(width='650px')
)


# ------------------------------------------------------------
# Play control
# ------------------------------------------------------------

play_control = Play(
    value=1,
    min=1,
    max=50,
    step=1,
    interval=800,
    description='Play',
    disabled=False,
    layout=Layout(width='115px')
)


# ------------------------------------------------------------
# Link Play and Time step
# ------------------------------------------------------------

jslink(
    (play_control, 'value'),
    (time_slider, 'value')
)


# ============================================================
# ANIMATION LOCK
# ============================================================

model_controls = [
    a1_slider,
    a2_slider,
    K_slider,
    ym1_slider,
    y0_slider,
    samples_slider,
    mode_buttons,
    time_slider
]


def set_animation_lock(locked):

    for widget in model_controls:

        widget.disabled = locked


def update_animation_lock(change):

    set_animation_lock(
        bool(change['new'])
    )


play_traits = play_control.traits()


if 'playing' in play_traits:

    play_state_trait = 'playing'


elif '_playing' in play_traits:

    play_state_trait = '_playing'


else:

    play_state_trait = None


if play_state_trait is not None:

    play_control.observe(
        update_animation_lock,
        names=play_state_trait
    )


set_animation_lock(
    False
)


# ============================================================
# SYNCHRONIZATION
# ============================================================

def update_time_range(change):

    new_maximum = change['new']

    time_slider.max = new_maximum

    play_control.max = new_maximum

    if time_slider.value > new_maximum:

        time_slider.value = new_maximum

    if play_control.value > new_maximum:

        play_control.value = new_maximum


samples_slider.observe(
    update_time_range,
    names='value'
)


# ------------------------------------------------------------
# Parameter changes restart the trajectory at n = 1
# ------------------------------------------------------------

def restart_time(change):

    time_slider.value = 1

    play_control.value = 1


a1_slider.observe(
    restart_time,
    names='value'
)


a2_slider.observe(
    restart_time,
    names='value'
)


K_slider.observe(
    restart_time,
    names='value'
)


ym1_slider.observe(
    restart_time,
    names='value'
)


y0_slider.observe(
    restart_time,
    names='value'
)


mode_buttons.observe(
    restart_time,
    names='value'
)


# ============================================================
# CONTROL LAYOUT
# ============================================================

controls_row_1 = HBox(
    [
        a1_slider,
        a2_slider,
        K_slider
    ],
    layout=Layout(
        width='950px',
        justify_content='space-between'
    )
)


controls_row_2 = HBox(
    [
        ym1_slider,
        y0_slider,
        samples_slider
    ],
    layout=Layout(
        width='900px',
        justify_content='space-between'
    )
)


controls_row_3 = HBox(
    [
        mode_buttons,
        play_control,
        time_slider
    ],
    layout=Layout(
        width='950px',
        justify_content='space-between',
        align_items='center'
    )
)


controls_box = VBox(
    [
        HTML("<div class='sg-controls-title'>Experiment controls</div>"),
        controls_row_1,
        controls_row_2,
        controls_row_3
    ],
    layout=Layout(
        width='960px',
        border='1px solid #d4d9df',
        padding='6px 8px',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Interactive plot
# ------------------------------------------------------------

widget_plot = interactive(
    plot_state_grid,
    a1=a1_slider,
    a2=a2_slider,
    K=K_slider,
    ym1=ym1_slider,
    y0=y0_slider,
    samples=samples_slider,
    time_step=time_slider,
    mode=mode_buttons
)


plot_output = widget_plot.children[-1]


plot_output.layout = Layout(
    width='auto',
    overflow='visible'
)


# ------------------------------------------------------------
# Final notebook layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        header_html,
        controls_box,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(main_layout)